In [ ]:
import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
from matplotlib import pylab
import os
import sys
anndata2ri.activate()
import yaml


In [ ]:
sc.set_figure_params(dpi=100, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (9, 9)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")
from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *


In [ ]:
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
anndata2ri.activate()
%load_ext rpy2.ipython
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Load data

In [ ]:
import scanpy as sc
import anndata as ad
import os
import glob




# Subset and pseudobulk

In [ ]:
adata = sc.read_h5ad(homeDir+"/Biermann_Atlas/adatas/Biermann_Atlas_curated.h5ad")

In [ ]:
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.set_figure_params(dpi=100, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (5, 5)
sc.pl.umap(adata, color="patient")

In [ ]:
# --- Imports & setup ----------------------------------------------------------
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.sparse import csr_matrix, issparse
import random

# Single RNG for reproducibility (change the seed if you like)
SEED = 1
rng = random.Random(SEED)
np_rng = np.random.default_rng(SEED)

# --- 1) Map each cell type to its "major" patients ---------------------------
# For each cell type (row), select patients whose counts are above the
# 70th percentile of *nonzero* counts in that row. If a row has no nonzeros,
# return an empty list for that cell type.
tab = pd.crosstab(adata.obs["celltypecurated"], adata.obs["patient"])

def major_patients_for_row(row: pd.Series, q: float = 70.0) -> list:
    pos = row[row > 0]
    if pos.empty:
        return []
    thr = np.percentile(pos.values, q)
    return row.index[row > thr].tolist()

PatientsMap = tab.apply(major_patients_for_row, axis=1)

# --- 2) Collect barcodes from those major patients per cell type -------------
selectedBCs = []
for ct in PatientsMap.index:
    keep_patients = PatientsMap[ct]
    if not keep_patients:
        continue
    mask = (adata.obs["celltypecurated"] == ct) & (adata.obs["patient"].isin(keep_patients))
    selectedBCs.extend(adata.obs_names[mask])

# Subset once; avoid needless .copy() unless you will mutate X/obs/var a lot
pbulkAdata = adata[selectedBCs].copy()

# --- 3) Compute per-cell-type minimum sample size across selected patients ---
# We need the minimum nonzero count of barcodes per (cell_type, patient),
# to know how many to sample per patient to balance.
ct_pt_counts = pd.crosstab(pbulkAdata.obs["celltypecurated"], pbulkAdata.obs["patient"])
minSamples = (
    ct_pt_counts.replace(0, np.nan)  # ignore zeros when taking the minimum
              .min(axis=1, skipna=True)  # per cell_type minimum across patients
              .fillna(0)                 # if a type is missing everywhere (unlikely)
              .astype(int)
              .to_dict()
)

# --- 4) Balanced sampling of barcodes across patients ------------------------
BalancedBCs = []
# Iterate patients present in the *subset* data (not the full adata)
for patient in pbulkAdata.obs["patient"].unique():
    adataPatient = pbulkAdata[pbulkAdata.obs["patient"] == patient]
    # For each cell type, sample up to minSamples[ct] barcodes from this patient
    for ct, n_take in minSamples.items():
        if n_take <= 0:
            continue
        typeBCs = adataPatient.obs_names[adataPatient.obs["celltypecurated"] == ct]
        if len(typeBCs) >= n_take:
            # Deterministic sampling with shared RNG
            BalancedBCs.extend(rng.sample(list(typeBCs), n_take))

# Final balanced subset
pbulkAdata = pbulkAdata[BalancedBCs].copy()

# Optional: quick check (comment out if noisy)
print(pd.crosstab(pbulkAdata.obs["celltypecurated"], pbulkAdata.obs["patient"]))

# --- 5) Library-size normalization ------------------------------------------
# Keep target_sum consistent with your downstream needs
sc.pp.normalize_total(pbulkAdata, target_sum=20_000)

# --- 6) Random aggregation into pseudo-bulk replicates -----------------------
def random_aggregation(
    _adata_group: ad.AnnData,
    group: str,
    cellStateObs: str,
    PseudoReplicates_per_group: int = 25,
    method: str = "random",
    seed: int = SEED,
) -> ad.AnnData:
    """
    For each cell state, split cells (round-robin on a shuffled list) into K partitions,
    and create one meta-cell per partition by averaging expression.
    Also store summed and median counts in layers (CSR sparse).
    """
    print(f"Random aggregation for sample '{group}' within '{cellStateObs}'")
    rng_local = random.Random(seed)

    states_adatas = []
    # Iterate unique states in requested obs
    for state in _adata_group.obs[cellStateObs].unique().tolist():
        state_adata = _adata_group[_adata_group.obs[cellStateObs] == state]
        group_cells = list(state_adata.obs_names)

        # Deterministic shuffle
        rng_local.shuffle(group_cells)

        # Round-robin partition into K pseudo-replicates
        partitions = [group_cells[i::PseudoReplicates_per_group] for i in range(PseudoReplicates_per_group)]

        # Materialize counts as a dense DataFrame only once per state/partition
        # (If X is sparse and large, this is still the simplest; for very large data,
        # replace with sparse-aware aggregations.)
        for k, cell_ids in enumerate(partitions):
            if not cell_ids:  # empty partition
                continue

            part = state_adata[cell_ids]

            # Get counts as DataFrame for convenient mean/sum/median ops
            # (genes as columns)
            counts_df = pd.DataFrame(
                part.X.toarray() if issparse(part.X) else part.X,
                index=part.obs_names,
                columns=part.var_names,
            )

            nCells = counts_df.shape[0]
            new_obs_name = f"{group}.{state}.{k}"

            # Average expression for meta-cell (1 x genes)
            meta_expr = counts_df.mean(axis=0).to_frame().T
            meta_expr.index = [new_obs_name]

            meta_adata = ad.AnnData(meta_expr)
            # Store alternative aggregations in layers (CSR)
            meta_adata.layers[f"{method}_summedCounts"] = csr_matrix(counts_df.sum(axis=0).values)
            meta_adata.layers[f"{method}_medianCounts"] = csr_matrix(counts_df.median(axis=0).values)

            # Annotate obs
            meta_adata.obs[cellStateObs] = state
            meta_adata.obs["nCells"] = nCells
            meta_adata.obs["group"] = group

            states_adatas.append(meta_adata)

    # Concatenate all meta-cells across states
    out = ad.concat(states_adatas, axis=0, join="outer", merge="same")
    # Ensure X is sparse CSR for memory efficiency
    if not issparse(out.X):
        out.X = csr_matrix(out.X)
    return out

# Preserve original obs columns that are constant across the subset
obs_before = pbulkAdata.obs.copy()

pbulk_meta = random_aggregation(
    pbulkAdata,
    group="Bierman",
    cellStateObs="celltypecurated",
    PseudoReplicates_per_group=25,
    method="random",
    seed=SEED,
)

# Re-attach obs columns that were constant across the original balanced subset
for col in obs_before.columns:
    uniques = obs_before[col].unique()
    if len(uniques) == 1:
        pbulk_meta.obs[col] = uniques[0]

# Save result
pbulk_meta.write_h5ad(homeDir+f"/Biermann_Atlas/adatas/AVGcounts_bycelltypecurated.h5ad")
